In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

import sys, os
sys.path.insert(0, os.path.abspath(".."))
from python.tokenizer import Tokenizer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
print(f"Using device: {device}")

In [3]:
text = ""
with open('../dataset/1 - A Game of Thrones.txt', 'r', encoding='utf-8', errors='ignore') as f:
    text += f.read()

from python.tokenizer import Tokenizer
tokenizer = Tokenizer.load("../tokenizer.json")
tokens = tokenizer.encode(text)

In [ ]:
class Head(nn.Module):

    def __init__(self, x_emb, head_emb):
        super().__init__()
        self.k = nn.Linear(x_emb, head_emb)
        self.q = nn.Linear(x_emb, head_emb)
        self.v = nn.Linear(x_emb, head_emb)

    def forward(self, x):
        k = self.k(x)                                   # (seq_length, head_emb)
        q = self.q(x)                                   # (seq_length, head_emb)
        v = self.v(x)                                   # (seq_length, head_emb)

        x = q @ k.transpose(-2, -1)                     # (seq_length, seq_length) we have not used q@k.T since it will be invalid operation in case of batches
        x = x/ pow(k.shape[-1], 0.5)
        x = F.softmax(x, dim=-1)                        # (seq_length, seq_length)
        x = x @ v                                       # (seq_length, head_emb)
        return x


In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, x_emb, heads_num, head_emb):
        super().__init__()
        self.heads_num = heads_num
        self.heads = nn.ModuleList([Head(x_emb, head_emb) for _ in range(heads_num)])
        self.proj = nn.Linear(x_emb, x_emb)

    def forward(self, tokens):
        x = torch.cat([head.forward(tokens) for head in self.heads], dim=-1)
        x = self.proj(x)
        return x
        

In [ ]:
class MultiheadBlock(nn.Module):

    def __init__ (self, heads_num, x_emb):
        super().__init__()
        self.layer_norm = nn.LayerNorm(x_emb)
        self.heads = MultiHeadAttention(x_emb, heads_num, x_emb//heads_num)

    def forward(self, tokens):
        x = self.layer_norm(tokens)
        x = self.heads(x)
        return tokens + x

In [ ]:
class FeedFwdBlock(nn.Module):

    def __init__(self, x_emb):
        super().__init__()
        self.layer = nn.Sequential(
        nn.Linear(x_emb, 4 * x_emb),
        nn.GELU(),
        nn.Linear(4 * x_emb, x_emb)
    )
        self.layer_norm = nn.LayerNorm(x_emb)

    def forward(self, tokens):
        x = self.layer_norm(tokens)
        x = self.layer(x)
        return tokens + x

In [ ]:
class EncoderArchitecture(nn.Module):

    def __init__(self, x_emb, heads_num):
        super().__init__()
        self.multihead = MultiheadBlock(heads_num, x_emb)
        self.feed_fwd = FeedFwdBlock(x_emb)

    def forward(self, tokens):
        x = self.multihead(tokens)
        x = self.feed_fwd(x)
        return x


In [ ]:
class Encoder(nn.Module):

    def __init__(self, vocab_size: int, x_emb: int, seq_len: int, heads_num: int, encoder_num: int):
        super().__init__()
        self.seq_len = seq_len
        self.MASK_TOKEN_ID = vocab_size+2
        self.look_up_table = nn.Parameter(torch.randn((vocab_size+3, x_emb)))
        self.postional_enc = nn.Parameter(torch.randn((seq_len, x_emb)))
        self.architecture = nn.ModuleList([EncoderArchitecture(x_emb, heads_num) for _ in range(encoder_num)])
        self.lm_head = nn.Linear(x_emb, vocab_size)
        self.optimizer = torch.optim.AdamW(self.parameters(), lr=1e-3)

    def forward(self, tokens, targets=None):
        tokens = torch.tensor(tokens)
        B, T = tokens.shape
        loss = None

        x = self.look_up_table[tokens] + self.postional_enc[torch.arange(T)]

        for block in self.architecture:
            x = block(x)

        logits = self.lm_head(x)       # (B, T, vocab_size)

        if targets is not None:
            targets = torch.tensor(targets)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def fit(self, tokens, epochs=100, batch_size=32):
        chunks = []
        for i in range(0, len(tokens) - self.seq_len + 1, self.seq_len + 1):
            chunks.append(tokens[i : i + self.seq_len + 1])

        last = tokens[len(chunks) * (self.seq_len + 1):]
        if len(last) > 0:
            last = last + [0] * (self.seq_len - len(last) + 1)
            chunks.append(last)

        x_chunks = []
        y_chunks = []
        for chunk in chunks:
            x = chunk.copy()
            y = [-100] * len(chunk)

            for i, token in enumerate(chunk):
                if random.random() < 0.15:
                    y[i] = token
                    x[i] = self.MASK_TOKEN_ID

        for epoch in range(epochs):
            combined = list(zip(x_chunks, y_chunks))
            random.shuffle(combined)
            x_shuffled, y_shuffled = zip(*combined)
            x, y = list(x_shuffled), list(y_shuffled)

            x_batch, y_batch = [], []
            for _ in range(batch_size):
                i = random.randint(0, len(x) - 1)
                x_batch.append(x[i])
                y_batch.append(y[i])

            output, loss = self.forward(x_batch, y_batch)
            print(f"Loss: {loss.item():.4f}, epoch: {epoch}")
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()


In [ ]:
test_encode = Encoder(vocab_size=30000, x_emb=64, seq_len=512, heads_num=2, encoder_num=6)
test_encode.fit(tokens=tokens, epochs=500, batch_size=32)

Loss: 10.8918, epoch: 0
Loss: 10.7185, epoch: 1
Loss: 10.5543, epoch: 2
Loss: 10.4199, epoch: 3
Loss: 10.2734, epoch: 4
Loss: 10.1355, epoch: 5
Loss: 9.9893, epoch: 6
Loss: 9.8217, epoch: 7
Loss: 9.6668, epoch: 8
Loss: 9.5115, epoch: 9
Loss: 9.3401, epoch: 10
Loss: 9.1872, epoch: 11
Loss: 9.0038, epoch: 12
Loss: 8.8080, epoch: 13
Loss: 8.6079, epoch: 14
Loss: 8.4383, epoch: 15
Loss: 8.2602, epoch: 16
Loss: 8.0654, epoch: 17
Loss: 7.8817, epoch: 18
Loss: 7.6926, epoch: 19
Loss: 7.6048, epoch: 20
Loss: 7.3736, epoch: 21
Loss: 7.2029, epoch: 22
Loss: 7.0485, epoch: 23
Loss: 6.9120, epoch: 24
Loss: 6.7403, epoch: 25
Loss: 6.6599, epoch: 26
Loss: 6.5353, epoch: 27
Loss: 6.4155, epoch: 28
Loss: 6.3360, epoch: 29
Loss: 6.2199, epoch: 30
Loss: 6.2804, epoch: 31
Loss: 6.0503, epoch: 32
Loss: 5.9683, epoch: 33
Loss: 5.9044, epoch: 34
Loss: 5.8731, epoch: 35
Loss: 5.8385, epoch: 36
Loss: 5.7636, epoch: 37
Loss: 5.7209, epoch: 38
Loss: 5.7018, epoch: 39
Loss: 5.6356, epoch: 40
Loss: 5.6468, epoch:

In [9]:
torch.cuda.empty_cache()

In [13]:
len(set(tokens))

3924

In [ ]:
tokens[:1000]

[65,
 14317,
 1005,
 5162,
 494,
 298,
 571,
 115,
 32,
 10,
 19021,
 10434,
 967,
 101,
 290,
 339,
 1833,
 792,
 290,
 4070,
 416,
 100,
 464,
 332,
 101,
 32,
 10,
 9377,
 410,
 101,
 9563,
 101,
 378,
 46,
 378,
 46,
 931,
 114,
 116,
 262,
 32,
 10,
 80,
 5883,
 76,
 79,
 71,
 85,
 69,
 32,
 10,
 34,
 1395,
 3660,
 632,
 100,
 3210,
 114,
 116,
 2510,
 391,
 405,
 14317,
 358,
 16326,
 270,
 326,
 291,
 101,
 6928,
 1038,
 115,
 1212,
 282,
 972,
 306,
 2081,
 114,
 107,
 552,
 614,
 100,
 291,
 595,
 46,
 299,
 2768,
 101,
 260,
 395,
 100,
 1738,
 792,
 115,
 552,
 101,
 32,
 10,
 939,
 324,
 340,
 32,
 10,
 34,
 2369,
 291,
 101,
 724,
 10003,
 104,
 1263,
 110,
 312,
 465,
 1001,
 114,
 12766,
 121,
 4195,
 4332,
 1319,
 270,
 3186,
 104,
 11409,
 323,
 291,
 101,
 7792,
 290,
 258,
 485,
 395,
 101,
 46,
 32,
 10,
 71,
 284,
 270,
 7694,
 100,
 425,
 116,
 1812,
 101,
 282,
 291,
 101,
 2510,
 287,
 46,
 383,
 979,
 115,
 416,
 10141,
 100,
 811,
 110,
 44,
 8808,
 323,
 1656